# Tools & workflows: giving your LLM the ability to *act* (running on LM Studio, 100% local)

**Goal of this notebook:** understand how an LLM goes from "just talks" to "can actually do things" — call a weather API, run a calculation, query a database — and how to chain several of those calls into a multi-step workflow.

We'll cover:
1. What a "tool" is to an LLM, and why frameworks aren't required to use one
2. Defining a tool and watching the model *ask* to call it
3. Closing the loop: running the tool and returning the result
4. Turning that into a reusable agent loop that can call multiple tools
5. A real multi-step workflow, where the second tool call depends on the first tool's result
6. Where frameworks (LangChain, CrewAI, etc.) fit in — and when you'd reach for one

> ⚠️ **This notebook runs entirely against your local LM Studio server — no API key, no cloud, no cost.** Because it talks to `localhost`, it must be run **locally** (e.g. in VS Code), not in Google Colab. Not every local model supports tool calling — this notebook uses a model that does (see Step 0).

## Which framework should I use?

**None, to start.** Every agent framework you'll hear about — LangChain, LlamaIndex, CrewAI, the OpenAI Agents SDK — is built on the exact same primitive: the model returns a structured "please call this function with these arguments" message instead of plain text, your code runs the real function, and you send the result back. That primitive is part of the OpenAI-compatible chat API itself (the `tools=` parameter), and LM Studio speaks it natively.

Learning the raw loop first means:
- You understand *exactly* what a framework is doing for you later, instead of it feeling like magic.
- The knowledge transfers to any framework — they all wrap this same loop.
- For a single agent with a handful of tools (most real projects), the raw loop is often all you need — a framework adds value once you need things like multi-agent coordination, long-term memory, or a large library of prebuilt tools.

So: this notebook builds the loop by hand. Part 6 maps it onto LangChain so you can see the translation once you already understand the mechanics.

## 🖥️ Step 0 — Set up LM Studio

1. Open LM Studio and make sure a **tool-calling capable** chat model is downloaded — not every model supports this. Models built on Llama 3.1+, Qwen, and Gemma 3/4 instruct families generally do.
2. Go to the **Developer** tab and click **Start Server** (default `http://localhost:1234`).
3. Run the cell below to list what's loaded, then set `CHAT_MODEL` further down to match exactly.

In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key="lm-studio")

print("Models available in LM Studio:")
for m in client.models.list().data:
    print(" -", m.id)

In [ ]:
CHAT_MODEL = "gemma-4-e2b-it-qat"  # any tool-calling capable chat model you loaded

## Part 1 — What is a "tool" to an LLM?

A tool is just a **JSON description of a function**: its name, what it does, and what arguments it takes. You send that description alongside your prompt. The model can't actually run code — all it can do is reply with *"call this tool with these arguments"* instead of a plain-text answer. Running the function and feeding the result back is entirely your app's job.

Let's describe one tool: a (fake, for this demo) weather lookup.

In [ ]:
def get_weather(city: str) -> dict:
    """Pretend weather API — swap this for a real HTTP call in a real app."""
    fake_db = {
        "paris": {"temp_c": 18, "condition": "cloudy"},
        "tokyo": {"temp_c": 26, "condition": "sunny"},
        "new york": {"temp_c": 21, "condition": "rainy"},
    }
    return fake_db.get(city.lower(), {"temp_c": 20, "condition": "unknown"})


weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Paris'"},
            },
            "required": ["city"],
        },
    },
}

Notice the shape: `name`, a plain-English `description` (the model uses this to decide *when* to call it — write these like documentation for a new teammate), and a `parameters` JSON Schema describing the arguments. This exact shape is what every framework's `@tool` decorator generates behind the scenes.

## Part 2 — Ask a question the model can't answer without the tool

We pass `tools=[weather_tool]` alongside the prompt. Watch what comes back — it's *not* a text answer.

In [ ]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Tokyo right now?"}],
    tools=[weather_tool],
    temperature=0,
)

message = response.choices[0].message
print("content:", repr(message.content))
print("tool_calls:", message.tool_calls)

`content` is empty — the model didn't try to guess the weather. Instead, `tool_calls` contains a request: call `get_weather` with `{"city": "Tokyo"}`. The model chose the tool and the arguments itself, purely from the tool's description and the JSON Schema — we never told it Tokyo would come up.

## Part 3 — Close the loop

Three steps: **run** the real function, **append** its result to the conversation as a `role="tool"` message (tagged with the same `tool_call_id`), and **ask again**. This time the model has the real data and can write a normal answer.

In [ ]:
import json

messages = [{"role": "user", "content": "What's the weather in Tokyo right now?"}]

# 1. Ask, with tools available
response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, tools=[weather_tool], temperature=0)
message = response.choices[0].message
messages.append(message)

# 2. Run every requested tool call and feed the result back
for tool_call in message.tool_calls:
    args = json.loads(tool_call.function.arguments)
    result = get_weather(**args)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result),
    })

# 3. Ask again — now with the real data in context
final = client.chat.completions.create(model=CHAT_MODEL, messages=messages, tools=[weather_tool], temperature=0)
print(final.choices[0].message.content)

That round trip — **ask → tool call → run it → tell the model → get the real answer** — is the entire mechanism. Everything else in this notebook is just making that loop more general.

## Part 4 — A reusable agent loop, with more than one tool

Real questions often need more than one tool in the same turn (or several turns in a row). Let's add a second tool and wrap the loop in a function that keeps going *until the model stops asking for tools* — that's what makes it a general-purpose **workflow** runner instead of a one-shot trick.

In [ ]:
def calculate(expression: str) -> dict:
    """Evaluate a basic arithmetic expression, e.g. '240 * 0.15'."""
    return {"result": eval(expression, {"__builtins__": {}})}


calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a basic arithmetic expression",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. '240 * 0.15'"},
            },
            "required": ["expression"],
        },
    },
}

TOOLBOX = {"get_weather": get_weather, "calculate": calculate}
TOOL_SCHEMAS = [weather_tool, calculator_tool]


def run_with_tools(user_message: str, max_steps: int = 5) -> str:
    messages = [{"role": "user", "content": user_message}]

    for _ in range(max_steps):
        response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0)
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content  # model is done — plain-text answer

        for tool_call in message.tool_calls:
            fn = TOOLBOX[tool_call.function.name]
            args = json.loads(tool_call.function.arguments)
            result = fn(**args)
            print(f"  → called {tool_call.function.name}({args}) = {result}")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})

    return "Gave up after too many steps."


print(run_with_tools("What's the weather in Paris, and what is 15% of 240?"))

One user turn, two tool calls, one coherent answer combining both. The model decided on its own that this question needed two different tools — we only gave it the toolbox.

## Part 5 — A *sequential* workflow

The example above ran two independent tool calls. A more realistic "workflow" is **sequential**: the second tool call needs the *output* of the first one, so the model can't request both at once — it has to see the first result before it can even form the second call. `run_with_tools` already handles this correctly, because it loops until the model is done rather than assuming one round of tool calls is enough.

In [ ]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Pretend FX API — swap for a real rates provider in production."""
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09}
    rate = rates.get((from_currency.upper(), to_currency.upper()), 1.0)
    return {"converted_amount": round(amount * rate, 2), "currency": to_currency.upper()}


currency_tool = {
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string", "description": "e.g. 'USD'"},
                "to_currency": {"type": "string", "description": "e.g. 'EUR'"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
}

TOOLBOX["convert_currency"] = convert_currency
TOOL_SCHEMAS.append(currency_tool)

print(run_with_tools(
    "I have 100 USD. Convert it to EUR, then tell me how many EUR 4.50 coffees "
    "I could buy with that amount."
))

Watch the printed steps: the model calls `convert_currency` first, gets the EUR amount back, and *only then* — now that it knows the number — calls `calculate` to divide by 4.50. It couldn't have formed that second call up front because it didn't know the converted amount yet. That dependency is what makes this a genuine multi-step **workflow** rather than two unrelated lookups.

## Part 6 — Where a framework like LangChain fits in

Once you understand the loop above, a framework is just naming its pieces for you:

| Raw loop (this notebook) | LangChain equivalent |
|---|---|
| Python function | `@tool`-decorated function |
| the `tools=[...]` list | a list of `Tool` objects bound to the model |
| our `while`/`for` loop | `AgentExecutor` (or LangGraph for more control) |
| `messages` list we appended to | the agent's memory / message history |

Reach for a framework when you need things the raw loop doesn't give you for free: multi-agent coordination, built-in memory strategies, a large library of ready-made tools (web search, SQL, file I/O), or observability/tracing. For one agent with a few tools you write yourself — like everything above — the raw loop is usually simpler to debug and has zero extra dependencies.

## Recap

- A **tool** is a JSON-described function: name, description, and a JSON Schema of its arguments.
- The model never runs code — it replies with a **tool call request**, and your app runs the real function.
- The core loop: **ask → tool call → run it → send the result back → ask again** — repeat until the model replies with plain text instead of a tool call.
- A **workflow** emerges naturally from that loop: give the model several tools and let it decide how many calls it needs and in what order, including calls that depend on earlier results.
- Frameworks (LangChain, CrewAI, LlamaIndex, the OpenAI Agents SDK) all wrap this same loop — learn it once, and every framework becomes readable instead of magic.

Next: try adding a third tool of your own, or set `max_steps` lower and see `run_with_tools` give up mid-workflow — a good way to feel *why* that safety limit exists before you hit it in a real app.